# LLM 실습 문제 (Practice)

각 셀에서 `## Write Answer` 부분을 채워보세요.

GPT from scratch: Tokenization → Dataset/DataLoader → Embedding → Attention → Transformer → Pretraining → Inference → Fine-tuning (LoRA / Classification / Instruction / DPO)

## Setup

In [ ]:
# [코드]
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 문제 1. BPE Tokenizer encode / decode

[문제]
Q 1) BPE 토크나이저를 사용하여 입력 문자열을 정수 토큰 ID 리스트로 인코딩하고 다시 텍스트로 디코딩합니다.

[요구사항]
- tokenizer.encode 메서드를 호출하여 특별 토큰 허용 옵션과 함께 인코딩
- tokenizer.decode 메서드를 호출하여 토큰 ID 시퀀스를 원본 문자열로 복원

In [ ]:
# [코드]
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.### 작성 필요 ###(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.### 작성 필요 ###(integers)
print(strings)

## 문제 2. Next-Token Prediction: 입력(x) / 정답(y) 쌍 만들기

[문제]
Q 2) Auto-regressive 언어 모델 학습을 위해 입력 토큰 시퀀스(x)에 대응하는 1 토큰 미래의 타깃 시퀀스(y)를 슬라이싱합니다.

[요구사항]
- enc_sample에서 context_size 크기의 입력 x를 추출
- 입력 x의 각 위치 바로 다음 시점의 토큰이 배치되도록 1 토큰 미래로 시프트된 타깃 y 슬라이싱

In [ ]:
# [코드]
enc_text = tokenizer.encode(raw_text)
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[### 작성 필요 ###:context_size+### 작성 필요 ###]

print(f"x: {x}")
print(f"y:      {y}")

## 문제 3. GPTDataset: sliding window로 (input, target) 생성

[문제]
Q 3) 긴 텍스트 토큰 ID 시퀀스로부터 sliding window 방식으로 (input, target) 배치 쌍을 생성하는 PyTorch Dataset을 완성합니다.

[요구사항]
- stride 간격으로 윈도우를 이동하며 max_length 길이의 입력 청크 추출
- 입력 청크보다 1 토큰 미래 시점인 max_length 길이의 타깃 청크 추출

In [ ]:
# [코드]
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[### 작성 필요 ### : ### 작성 필요 ### + max_length]
            target_chunk = token_ids[### 작성 필요 ### + ### 작성 필요 ### : ### 작성 필요 ### + max_length + ### 작성 필요 ###]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[### 작성 필요 ###], self.target_ids[### 작성 필요 ###]

## 문제 4. Token Embedding + Positional Embedding

[문제]
Q 4) 토큰 임베딩 레이어와 학습 가능한 위치 임베딩(Positional Embedding) 레이어를 생성하고 두 텐서를 결합합니다.

[요구사항]
- context_length 크기의 위치 임베딩 nn.Embedding 레이어 정의
- 0부터 seq_len-1까지의 정수 인덱스 텐서를 위치 임베딩 레이어에 전달하여 토큰 임베딩과 원소별 합산

In [ ]:
# [코드]
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

pos_embedding_layer = torch.nn.Embedding(### 작성 필요 ###, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(### 작성 필요 ###))
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

## 문제 5. Causal (Masked) Self-Attention

[문제]
Q 5) 인과적 어텐션 마스크(Causal Mask)가 적용된 Self-Attention 모듈의 Q, K, V 선형 투영 레이어를 초기화합니다.

[요구사항]
- 입력 차원(d_in)을 출력 차원(d_out)으로 변환하는 nn.Linear 투영층 3개(W_query, W_key, W_value) 정의
- 인자로 전달된 qkv_bias 플래그를 편향(bias) 옵션에 전달

In [ ]:
# [코드]
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.W_query = nn.### 작성 필요 ###(### 작성 필요 ###, ### 작성 필요 ###, bias=qkv_bias)
        self.W_key   = nn.### 작성 필요 ###(### 작성 필요 ###, ### 작성 필요 ###, bias=qkv_bias)
        self.W_value = nn.### 작성 필요 ###(### 작성 필요 ###, ### 작성 필요 ###, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.### 작성 필요 ###(### 작성 필요 ###)
        queries = self.### 작성 필요 ###(### 작성 필요 ###)
        values = self.### 작성 필요 ###(### 작성 필요 ###)

        attn_scores = ### 작성 필요 ### @ ### 작성 필요 ###.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            ### 작성 필요 ###
        )
        attn_weights = torch.### 작성 필요 ###(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ ### 작성 필요 ###

        return context_vec

## 문제 6. Multi-Head Attention (head 분할/결합)

[문제]
Q 6) Multi-Head Attention에서 Q, K, V 텐서를 여러 헤드로 분할(split) 및 전치(transpose)하여 병렬 연산 형상으로 변환합니다.

[요구사항]
- 총 출력 차원(d_out)을 헤드 수(num_heads)로 나누어 head_dim 계산
- (batch, seq_len, num_heads, head_dim)으로 뷰 변환 후 헤드 축과 시퀀스 축을 전치하여 (batch, num_heads, seq_len, head_dim) 형상 생성

In [ ]:
# [코드]
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = ### 작성 필요 ###

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = ### 작성 필요 ###
        values = ### 작성 필요 ###
        queries = ### 작성 필요 ###

        attn_scores = ### 작성 필요 ###

        mask_bool = self.### 작성 필요 ###

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = ### 작성 필요 ###
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = ### 작성 필요 ###
        context_vec = self.out_proj(context_vec)

        return context_vec

## 문제 7. LayerNorm

[문제]
Q 7) Transformer용 LayerNorm 모듈에서 마지막 은닉 차원(dim=-1) 기준의 평균과 분산을 계산하여 정규화합니다.

[요구사항]
- 마지막 차원에 대해 keepdim=True 옵션으로 평균(mean)과 분산(var) 계산
- 표준편차 분모에 수치적 안정성을 위한 epsilon(self.eps) 추가

In [ ]:
# [코드]
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = ### 작성 필요 ###
        var = ### 작성 필요 ###
        norm_x = (x - ### 작성 필요 ###) / torch.sqrt(### 작성 필요 ### + self.eps)
        return self.### 작성 필요 ### * norm_x + self.### 작성 필요 ###

## 문제 8. FeedForward (GPT-style MLP)

[문제]
Q 8) GPT-style Transformer의 FeedForward(MLP) 블록을 4배 확장 선형층, GELU 활성화, 축소 선형층으로 구성합니다.

[요구사항]
- 임베딩 차원을 4배 확장(4 * emb_dim)하는 선형 레이어 생성
- 사전 정의된 GELU 활성화 함수 통과 후 다시 원래 차원(emb_dim)으로 축소하는 선형 레이어 연결
- ※ 사전 정의 컴포넌트: GELU() 클래스가 제공됩니다.

In [ ]:
# [코드]
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], ### 작성 필요 ### * cfg["emb_dim"]),
            ### 작성 필요 ###(),
            nn.Linear(### 작성 필요 ### * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

## 문제 9. Transformer Block forward (Residual 연결 포함)

[문제]
Q 9) 잔차 연결(Residual Connection)과 드롭아웃을 포함하는 Transformer Block의 순전파(forward) 로직을 완성합니다.

[요구사항]
- 입력 x에 대해 첫 번째 서브블록(어텐션) 통과 후 잔차 덧셈 (shortcut connection) 수행
- 정규화 및 두 번째 서브블록(FeedForward) 통과 후 다시 이전 결과와 잔차 덧셈 수행
- ※ 사전 정의 컴포넌트: LayerNorm(emb_dim), MultiHeadAttention(...), FeedForward(cfg) 가 제공됩니다.

In [ ]:
# [코드]
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.### 작성 필요 ###(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.### 작성 필요 ###(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

## 문제 10. GPTModel (전체 아키텍처 조립 + forward)

[문제]
Q 10) 토큰/위치 임베딩과 트랜스포머 블록들을 조합하여 전체 GPTModel 클래스를 완성합니다.

[요구사항]
- 설정 사전(cfg)의 vocab_size, emb_dim, context_length를 참조하여 임베딩 및 출력 선형 헤드 초기화
- forward에서 입력 인덱스를 임베딩하고 블록들을 순차 통과시킨 후 최종 logits 반환
- ※ 사전 정의 컴포넌트: TransformerBlock(cfg), LayerNorm(emb_dim) 이 제공됩니다.

In [ ]:
# [코드]
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["### 작성 필요 ###"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["### 작성 필요 ###"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["### 작성 필요 ###"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(### 작성 필요 ###)

        x = tok_embeds + ### 작성 필요 ###
        x = self.drop_emb(x)
        x = self.### 작성 필요 ###(x)
        x = self.final_norm(x)
        logits = self.### 작성 필요 ###(x)

        return logits

## 문제 11. Greedy Text Generation Loop

[문제]
Q 11) 학습된 GPT 모델로부터 탐욕적 방식(Greedy Decoding)으로 다음 토큰을 순차 생성하는 루프를 구현합니다.

[요구사항]
- 기울기 계산 비활성화(torch.no_grad) 컨텍스트 진입
- 마지막 시점의 로짓 텐서에 대해 argmax를 취하여 가장 확률이 높은 다음 토큰 선택
- 선택된 토큰 ID를 기존 입력 시퀀스 끝에 연결하여 다음 스텝 입력으로 갱신

In [ ]:
# [코드]
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.### 작성 필요 ###():
            logits = model(idx_cond)
        logits = logits[:, ### 작성 필요 ###, :]

        idx_next = torch.### 작성 필요 ###(logits, dim=-1, keepdim=True)
        idx = torch.### 작성 필요 ###((idx, idx_next), dim=1)

    return idx

## 문제 12. Pretraining Loss (Cross Entropy over Vocab)

[문제]
Q 12) 어휘집 전체에 대한 다중 클래스 분류 문제로서 언어 모델의 교차 엔트로피 손실(Cross-Entropy Loss)을 계산합니다.

[요구사항]
- 로짓 텐서의 배치와 시퀀스 차원을 평탄화(flatten)하여 (B*T, vocab_size) 형태로 변환
- 타깃 텐서를 1차원 (B*T,) 벡터로 평탄화하여 F.cross_entropy에 전달

In [ ]:
# [코드]
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.### 작성 필요 ###(
        logits.### 작성 필요 ###(0, 1),
        target_batch.### 작성 필요 ###()
    )
    return loss

## 문제 13. PyTorch 학습 루프 (모드 전환 + zero_grad / backward / step)

[문제]
Q 13) 에폭 단위 학습 루프에서 모델을 train 모드로 전환하고 역전파 및 최적화를 수행합니다.

[요구사항]
- 에폭 시작 시 model.train() 모드 전환
- 배치 손실 계산 후 optimizer.zero_grad(), loss.backward(), optimizer.step() 실행
- ※ 사전 정의 함수: calc_loss_batch, evaluate_model 이 제공됩니다.

In [ ]:
# [코드]
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.### 작성 필요 ###()

        for input_batch, target_batch in train_loader:
            optimizer.### 작성 필요 ###()

            loss = ### 작성 필요 ###(input_batch, target_batch, model, device)

            loss.### 작성 필요 ###()

            optimizer.### 작성 필요 ###()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

    return train_losses, val_losses, track_tokens_seen

## 문제 14. Text ↔ Token IDs 변환 (batch 차원 추가/제거)

[문제]
Q 14) 단일 텍스트를 모델 입력 규격에 맞게 1차원 토큰 텐서에서 2차원 배치 텐서 (1, seq_len)로 변환하고 생성 결과를 다시 1차원으로 복원합니다.

[요구사항]
- torch.tensor.unsqueeze(0)를 사용하여 맨 앞 배치 차원 추가
- 모델 출력 텐서에서 squeeze(0)를 사용하여 배치 차원 제거

In [ ]:
# [코드]
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).### 작성 필요 ###(0)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.### 작성 필요 ###(0)
    return tokenizer.decode(flat.tolist())

## 문제 15. LoRA (Low-Rank Adaptation) 레이어

[문제]
Q 15) 사전학습 가중치를 고정하고 저차원 행렬 곱 (B @ A)으로 델타를 학습하는 LoRALayer 모듈의 순전파를 완성합니다.

[요구사항]
- 원래의 고정 선형 레이어 출력에 스케일 팩터 (alpha / rank)가 적용된 저차원 어댑터 출력 (x @ A @ B)을 합산

In [ ]:
# [코드]
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, ### 작성 필요 ###))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        self.B = nn.Parameter(torch.zeros(### 작성 필요 ###, out_dim))

        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.### 작성 필요 ### / self.rank) * (x @ self.### 작성 필요 ### @ self.### 작성 필요 ###)
        return x


class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.### 작성 필요 ###(x)


def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.### 작성 필요 ###):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

## 문제 16. LoRA 적용 (전체 freeze → LoRA만 학습)

[문제]
Q 16) 사전학습 모델의 모든 기본 파라미터를 동결(freeze)하고 선형 레이어를 LoRA 레이어로 교체합니다.

[요구사항]
- 모델 전체 파라미터의 requires_grad를 False로 설정하여 가중치 동결
- ※ 사전 정의 함수: 앞서 구현된 LoRA 교체 헬퍼 replace_linear_with_lora(model, rank, alpha) 가 제공됩니다.

In [ ]:
# [코드]
for param in model.parameters():
    param.requires_grad = ### 작성 필요 ###

### 작성 필요 ###(model, rank=LORA_RANK, alpha=LORA_ALPHA)

model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 학습 가능 LoRA 파라미터 수: {total_params:,}")

## 문제 17. 분류 Fine-tuning (마지막 토큰 logit + 선택적 freeze)

[문제]
Q 17) GPT 모델을 텍스트 분류기로 사용하기 위해 출력 헤드를 교체하고 마지막 트랜스포머 블록만 학습 가능하도록 설정합니다.

[요구사항]
- 모델의 모든 기본 파라미터를 동결한 후 마지막 트랜스포머 블록(model.trf_blocks[-1])과 새 분류 헤드의 requires_grad를 활성화
- 출력 헤드를 분류 클래스 수(num_classes)에 맞는 새로운 nn.Linear로 교체

In [ ]:
# [코드]
for param in model.parameters():
    param.requires_grad = ### 작성 필요 ###

torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=### 작성 필요 ###)

model.to(device)

for param in model.trf_blocks[### 작성 필요 ###].parameters():
    param.requires_grad = ### 작성 필요 ###

for param in model.final_norm.parameters():
    param.requires_grad = ### 작성 필요 ###


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, ### 작성 필요 ###, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

## 문제 18. Instruction Finetuning용 Collate 함수 (Shift + Padding + Mask)

[문제]
Q 18) 명령어 파인튜닝(Instruction Finetuning) 데이터셋을 위한 collate_fn에서 패딩 및 손실 계산 마스크(ignore_index)를 적용합니다.

[요구사항]
- 배치 내 최대 길이에 맞춰 pad_token_id로 입력 텐서 패딩
- 프롬프트 영역 및 패딩 토큰 위치의 타깃 라벨을 ignore_index(-100)로 마스킹하여 손실 계산에서 제외

In [ ]:
# [코드]
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [### 작성 필요 ###]

        padded = new_item + [### 작성 필요 ###] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:### 작성 필요 ###])
        targets = torch.tensor(padded[### 작성 필요 ###:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[### 작성 필요 ###:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

## 문제 19. DPO Log Probability & Loss

[문제]
Q 19) DPO(Direct Preference Optimization) 손실 함수에서 선호 응답과 비선호 응답의 로그 확률 차이를 계산합니다.

[요구사항]
- 선호 응답(chosen)과 비선호 응답(rejected)에 대해 참조 모델과 정책 모델의 로그 확률 비(log ratio) 산출
- 두 로그 확률 비의 차이에 beta를 곱하고 -F.logsigmoid를 적용하여 DPO 손실 계산

In [ ]:
# [코드]
def compute_logprobs(logits, labels, selection_mask=None):
    labels = labels[:, ### 작성 필요 ###:].clone()
    logits = logits[:, :### 작성 필요 ###, :]

    log_probs = F.log_softmax(logits, dim=-1)

    selected_log_probs = torch.gather(
        input=### 작성 필요 ###,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)


def compute_dpo_loss(model_chosen_logprobs, model_rejected_logprobs,
                     reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1):
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs

    logits = model_logratios - reference_logratios

    losses = -F.### 작성 필요 ###(beta * logits)

    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

## 문제 20. RAG: 임베딩 코사인 유사도 기반 Top-K 문서 검색

[문제]
Q 20) 쿼리 벡터와 문서 임베딩 행렬 간 코사인 유사도를 산출하고 상위 k개 문서의 인덱스를 반환합니다.

[요구사항]
- F.cosine_similarity를 사용하여 쿼리(1, D)와 문서 행렬(N, D) 간의 마지막 차원(dim=-1) 유사도 계산
- torch.topk를 이용해 유사도가 가장 높은 상위 k개 문서의 인덱스(indices) 추출

In [ ]:
# [코드]
def retrieve_top_k(query_emb: torch.Tensor, doc_embs: torch.Tensor, k: int = 3):
    # query_emb: (1, D), doc_embs: (N, D)
    similarities = ### 작성 필요 ###
    top_scores, top_indices = ### 작성 필요 ###
    return top_indices, top_scores

## 문제 21. RAG: LlamaIndex 기반 VectorStoreIndex & Query Engine 구성

[문제]
Q 21) LlamaIndex를 활용하여 문서 목록을 VectorStoreIndex로 인덱싱하고, 상위 top_k 문서를 참조하는 Query Engine으로 질의 응답을 실행합니다.

[요구사항]
- VectorStoreIndex.from_documents를 호출하여 문서 인덱스 생성
- index.as_query_engine 호출 시 similarity_top_k 인자 전달
- 생성된 query_engine의 query 메서드로 사용자 질의(question) 실행
- ※ 사전 정의 라이브러리: llama_index.core.VectorStoreIndex 가 제공됩니다.

In [ ]:
# [코드]
from llama_index.core import VectorStoreIndex

def run_llamaindex_rag(documents, question: str, top_k: int = 3):
    index = ### 작성 필요 ###
    query_engine = ### 작성 필요 ###
    response = ### 작성 필요 ###
    return response

## 문제 22. RAG: 검색된 컨텍스트 기반 증강 프롬프트(Augmented Prompt) 조립

[문제]
Q 22) 검색된 문서 청크 리스트(chunks)를 하나의 컨텍스트 문자열로 결합하고 RAG 프롬프트 템플릿에 문맥과 질문을 바인딩합니다.

[요구사항]
- chunks 리스트의 각 문자열을 두 줄 개행('\n\n')으로 결합하여 단일 context 문자열 생성
- f-string 템플릿의 [문맥] 영역에 context 변수를, [질문] 영역에 question 변수를 삽입

In [ ]:
# [코드]
def build_rag_prompt(chunks: list[str], question: str) -> str:
    context = ### 작성 필요 ###
    prompt = f"""다음 [문맥]만을 참고하여 [질문]에 답하세요.

[문맥]
{### 작성 필요 ###}

[질문]
{### 작성 필요 ###}

[답변]"""
    return prompt

## 문제 23. MCP (Model Context Protocol): FastMCP 기반 Tool 등록 및 구현

[문제]
Q 23) FastMCP 서버 인스턴스를 초기화하고, 문서 검색 도구(search_docs)를 MCP Tool로 등록하는 코드를 완성합니다.

[요구사항]
- FastMCP 클래스를 지정된 서버 이름('RAGServer')으로 초기화
- 함수 상단에 MCP 도구 등록용 데코레이터(@mcp.tool()) 적용
- ※ 사전 정의 라이브러리: mcp.server.fastmcp.FastMCP 가 제공됩니다.

In [ ]:
# [코드]
from mcp.server.fastmcp import FastMCP

mcp = ### 작성 필요 ###

@### 작성 필요 ###
def search_docs(query: str, top_k: int = 3) -> list[str]:
    """지식 베이스에서 쿼리와 관련된 상위 top_k 문서를 검색합니다."""
    return [f"Result {i+1} for {query}" for i in range(top_k)]